# Legal RAG Bench: Hierarchy Construction and Analysis

Part 1 prepares original-child and merged-parent corpora for controlled retrieval comparisons.
This notebook presents actual corpus analysis and validated context-construction examples.
It does not report BM25/hybrid scores or reproduce the 44 → 56 claim.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'hierarchy').exists() and (ROOT / 'legal_rag_hierarchy').exists():
    ROOT = ROOT / 'legal_rag_hierarchy'
summary = json.loads((ROOT / 'results/summary.json').read_text(encoding='utf-8'))
read_jsonl = lambda path: [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
children = read_jsonl(ROOT / 'data/children.jsonl')
parents = read_jsonl(ROOT / 'data/parents.jsonl')
queries = read_jsonl(ROOT / 'data/queries.jsonl')

## 1. Decode and verify the hierarchy

`1.1-c1-s1` → c-parent `1.1-c1` → broader section `1.1`.
Removing only `-sN` gives 2,785 parents; removing both suffixes gives 724 section groups.
The authors describe hierarchical and semantic splitting but do not formally define these ID
components in the inspected documentation. We use ID-based grouping supported by consistent
titles, contiguous numeric sequences, and text inspection. These are reconstructed groups,
not recovered Word files. Some groups include nested headings.

In [2]:
display(pd.DataFrame([
    ('Original children', summary['n_children']), ('Merged c-parents', summary['n_parents']),
    ('Broader sections', summary['n_sections']), ('Parents with one child', summary['singleton_parents']),
    ('Maximum children per parent', summary['children_per_parent']['max']),
], columns=['Unit', 'Count']))

,Unit,Count
0,Original children,4876
1,Merged c-parents,2785
2,Broader sections,724
3,Parents with one child,2008
4,Maximum children per parent,39


## 2. Inspect real examples

Introductory Remarks continues its numbering across two children. Excusing Jurors has a
heading-only child, two substantive children, and an update notice. The largest group has
39 children and contains nested subheadings. We preserve all these pieces in numeric order.

In [3]:
example_ids = ['1.1-c1', '1.2-c2', '4.12-c7']
display(pd.DataFrame([{'Parent': p['id'], 'Title': p['title'], 'Children': len(p['child_ids']),
                       'First child': p['child_ids'][0], 'Last child': p['child_ids'][-1]}
                      for p in parents if p['id'] in example_ids]))
display(pd.DataFrame([{'ID': c['id'], 'Parent': c['parent_id'], 'Text preview': c['text'][:110]}
                     for c in children if c['parent_id'] in ['1.1-c1', '1.2-c2']]))

,Parent,Title,Children,First child,Last child
0,1.1-c1,1.1 Introductory Remarks,2,1.1-c1-s1,1.1-c1-s2
1,1.2-c2,1.2 Excusing Jurors,4,1.2-c2-s1,1.2-c2-s4
2,4.12-c7,4.12 When to Give an Identification Evidence W...,39,4.12-c7-s1,4.12-c7-s39


,ID,Parent,Text preview
0,1.1-c1-s1,1.1-c1,# 1.1 Introductory Remarks\n\n\n\n1. A number ...
1,1.1-c1-s2,1.1-c1,4. While the Juries Commissioner provides some...
2,1.2-c2-s1,1.2-c2,## Excusing Jurors
3,1.2-c2-s2,1.2-c2,1. After providing the jury panel with this in...
4,1.2-c2-s3,1.2-c2,7. There is no absolute right of an accused pe...
5,1.2-c2-s4,1.2-c2,Last updated: 11 July 2018


## 3. Ordering, overlap, and footnotes

All groups have consistent titles and gap-free numbering. We sort numerically, including s10
after s9. All child texts round-trip through parent character offsets. No text is deleted.
The overlap check tests exact suffix/prefix characters and case-normalized whitespace words;
it does not detect paraphrases. Footnote fields are excluded from both retrieval texts and
preserved as metadata because many adjacent children repeat them.

In [4]:
display(pd.DataFrame([
    ('Adjacent boundaries checked', summary['adjacent_boundaries_checked']),
    ('Any normalized word overlap', summary['boundaries_with_any_normalized_word_overlap']),
    ('Review flags (30 characters or 5 words)', summary['overlap_review_flags']),
    ('Groups requiring reordering', summary['parents_reordered']),
    ('Adjacent pairs repeating footnotes', summary['repeated_adjacent_footnotes']),
    ('Every child span verified', summary['all_spans_round_trip']),
], columns=['Check', 'Result']))

,Check,Result
0,Adjacent boundaries checked,2091
1,Any normalized word overlap,0
2,Review flags (30 characters or 5 words),0
3,Groups requiring reordering,0
4,Adjacent pairs repeating footnotes,1092
5,Every child span verified,True


## 4. Token distributions

Counts use the pinned MiniLM WordPiece tokenizer with truncation disabled. Content counts
exclude special tokens; the limit check includes them. This model's sentence-transformer
configuration limits inputs to 256 tokens. Counts differ from the dataset's Kanon tokenizer.
If Part 2 uses another embedding model, it must check that tokenizer and limit separately.

In [5]:
display(pd.read_csv(ROOT / 'results/token_summary.csv').round(2))
lengths = pd.read_csv(ROOT / 'results/unit_lengths.csv')
lengths['size_band'] = pd.cut(lengths['content_tokens'], bins=[0,254,512,1024,2048,4096,20000],
                             labels=['1–254','255–512','513–1024','1025–2048','2049–4096','4097+'])
display(pd.crosstab(lengths['size_band'], lengths['variant'], dropna=False))
display(lengths[lengths.variant == 'parent'].nlargest(5, 'content_tokens'))

,variant,count,min,mean,median,p95,max,model_limit_including_specials,over_model_limit_count,over_model_limit_fraction
0,child,4876,3,288.90,286.0,513.0,576,256,2684,0.55
1,parent,2785,17,505.82,299.0,1651.8,12632,256,1556,0.56


variant,child,parent
size_band,,
1–254,2192,1229
255–512,2437,756
513–1024,247,492
1025–2048,0,215
2049–4096,0,74
4097+,0,19


,variant,id,section_id,words,characters,content_tokens,tokens_with_specials,over_model_limit,n_children,size_band
5168,parent,4.12-c7,4.12,9196,54026,12632,12634,True,39,4097+
5210,parent,4.14-c2,4.14,5570,32823,7686,7688,True,24,4097+
4909,parent,1.7-c4,1.7,4717,26669,6849,6851,True,22,4097+
5002,parent,3.9-c1,3.9,4339,24230,6228,6230,True,20,4097+
7367,parent,8.9-c7,8.9,4169,24977,5859,5861,True,18,4097+


## 5. Corpora and gold mappings for Part 2

Children keep original IDs and text. Parents add ordered child IDs and exact character spans.
Questions retain original gold child IDs and derived parent IDs; these are labels, not retrieval
inputs. The existing 20-dev/80-test split is preserved. Three parents contain gold children on
both sides: parameter tuning is query-held-out, not parent-disjoint.

In [6]:
display(pd.DataFrame([{'Query': q['id'], 'Gold child': q['gold_child_ids'][0],
                       'Gold parent': q['gold_parent_ids'][0], 'Split': q['split']} for q in queries[:5]]))
display(pd.Series(summary['split_counts'], name='Questions').to_frame())
print('Shared gold parent IDs:', summary['shared_gold_parent_ids_between_dev_test'])
display(pd.read_csv(ROOT / 'data/child_spans.csv').head(6))

,Query,Gold child,Gold parent,Split
0,1,1.2-c2-s2,1.2-c2,test
1,2,1.5-c5-s1,1.5-c5,test
2,3,1.5-c6-s1,1.5-c6,dev
3,4,1.5-c7-s2,1.5-c7,test
4,5,1.5-c8-s1,1.5-c8,test


,Questions
test,80
dev,20


Shared gold parent IDs: ['7.1.3-c1', '7.4.9-c5', '7.6.1-c3']


,child_id,parent_id,section_id,s_index,start_char,end_char
0,1.1-c1-s1,1.1-c1,1.1,1,0,1459
1,1.1-c1-s2,1.1-c1,1.1,2,1461,3798
2,1.2-c1-s1,1.2-c1,1.2,1,0,1822
3,1.2-c2-s1,1.2-c2,1.2,1,0,18
4,1.2-c2-s2,1.2-c2,1.2,2,20,2249
5,1.2-c2-s3,1.2-c2,1.2,3,2251,2931


## 6. Why parent discovery and delivered evidence differ

The following is a **manually selected sibling example**, not model retrieval. The seed is a
heading-only sibling of the gold child. Direct child context misses the gold text; expansion
includes it, while a small context budget removes it again. Part 2 should report original-child
evidence in the actual assembled context, as well as parent membership and context size.

In [7]:
example = pd.read_csv(ROOT / 'results/expansion_example.csv')
display(example[['seed_child_id','mode','k','token_budget','ranked_unit_hit_at_k',
                 'gold_child_hit_in_context','gold_body_token_coverage','returned_content_tokens']])

,seed_child_id,mode,k,token_budget,ranked_unit_hit_at_k,gold_child_hit_in_context,gold_body_token_coverage,returned_content_tokens
0,1.2-c2-s1,child,1,NaN,0.0,0.0,0.000000,8
1,1.2-c2-s1,child_to_parent,1,NaN,0.0,1.0,1.000000,670
2,1.2-c2-s1,child_to_parent,1,10.0,0.0,0.0,0.003984,10


## 7. Prefix visibility diagnostic

Here the correct parent is supplied directly from the label. We check whether the original
gold child's full body survives the first 254 or 2,048 content tokens. The smaller prefix is a
MiniLM-sized content allowance after special tokens; it is a text-visibility check, not a
simulation of embedding quality. These counts are **not retrieval accuracy**.

In [8]:
visibility = pd.read_csv(ROOT / 'results/gold_visibility.csv')
display(visibility.groupby('prefix_budget_content_tokens').agg(
    Questions=('query_id','count'), Full_gold_body_visible=('full_gold_body_in_prefix','sum')))

,Questions,Full_gold_body_visible
prefix_budget_content_tokens,,
254,100,9
2048,100,89


## 8. Part 2 experiment contract

Compare the same BM25/hybrid settings on child and parent corpora. For child-to-parent
expansion, reuse the child rankings, take top-k seeds, and deduplicate parents without
backfilling. Evaluate k=1,5,10 under both unbounded context and a common token budget.

Use `hierarchy.evaluate` for saved predictions. It reports rank-unit hits, original-gold-child
inclusion after truncation, gold-body token coverage, and context sizes. Body coverage is not
answer-fact completeness. Timing and actual retrieval are Part 2's responsibility.

The 44 → 56 result remains unverified: its metric, denominator, corpus grouping, model,
weights, k, query split, and token budget must first be identified.

See PART2_HANDOFF.md for the adapter contract and runnable evaluation commands.
Sources: [dataset](https://huggingface.co/datasets/isaacus/legal-rag-bench),
[paper](https://arxiv.org/html/2603.01710v1#S3),
[tokenizer/model](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).